# Notebook: Introduction to Prompting Strategies for Large Language Models (LLMs)

This notebook introduces various prompting strategies that can help improve the performance of Large Language Models (LLMs).

## 📚 Sources

- [Prompting Guide](https://www.promptingguide.ai/techniques)
- [Python Package `openai`](https://github.com/openai/openai-python)

---

Have fun exploring and experimenting! 🤗

We use the `openai` package to interact with an LLM. This package allows communication with both the OpenAI API and other LLMs that offer a compatible API.

In this notebook, we'll talk to an LLM server running Ollama. Make sure you have access to the endpoint given to you (see README.md / setup instructions).

In [36]:
# Now let's import the package
from openai import OpenAI

In [37]:
# Let's define the API URL and the models we'll use. Note the hint above!
LLM_URL = "http://132.199.138.16:11434/v1"

LLM_MODEL = "gemma3:4b"       # small, fast model
LLM_REASONING = "gemma4:26b"  # larger, more capable model

We'll use two different models throughout this notebook:

- **`LLM_MODEL`** (`gemma3:4b`) — a small, fast model. It understands both the modern **Chat Completions API** and the older, raw **Completions API** (more on the difference between the two in a second), which makes it a good default for trying out different prompting techniques.
- **`LLM_REASONING`** (`gemma4:26b`) — a larger, more capable model. It is a **Mixture-of-Experts (MoE)** model: internally, it is built from many smaller, specialized "expert" sub-networks, but for any given token, only a handful of these experts are actually activated — not the whole model. This means it can have a huge total number of parameters (and correspondingly strong performance) while staying much cheaper to run than a regular ("dense") model of the same total size, which would use all of its parameters for every single token.

  One consequence of how `gemma4:26b` was trained is that it only ever saw **chat-style** conversations (messages with a `role` and `content`) during training — never plain, non-chat text continuation. So we will only ever use `LLM_REASONING` through the Chat Completions API, never through the raw Completions API.

In [38]:
client = OpenAI(
    base_url=LLM_URL,
    api_key="ollama",
)

## 1. Using the Chat Completions API

Most modern LLM providers (OpenAI, Ollama, ...) expose a **Chat Completions API**. Instead of sending a single block of text, you send a list of `messages`, where each message has a `role` (`"system"`, `"user"`, or `"assistant"`) and `content`. This format was designed for instruction-tuned, conversational models, and it's what you'll use for almost everything in this course (and in practice).

The generated reply is available at `response.choices[0].message.content`.

In [39]:
response = client.chat.completions.create(
    model=LLM_REASONING,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.0,
)

print(response.choices[0].message.content)

The capital of France is **Paris**.


Notice the `response.choices[0]` at the end. `choices` is a **list**, because the API can return several alternative completions for a single request — you control how many via the `n` parameter (default `n=1`). So by default `choices` has exactly one entry, and `choices[0]` simply means "the one answer". Each entry also carries its own `finish_reason` (e.g. `"stop"` if the model naturally finished, or `"length"` if it got cut off by `max_tokens`) telling you why generation stopped. We'll always use `n=1` in this notebook, but keep this in mind — it's why the `[0]` is there even though there's only one result.

#### What's actually sent to the model?

The `messages` list you send doesn't magically become a conversation — behind the scenes, the server takes it and renders it into one single raw text prompt using a **chat template**, then feeds that raw text to the model, essentially the same way as the raw Completions API in Section 2. Every model family defines its own chat template with its own special tokens for marking turns. Here's what our very first example above (the question and its answer) actually turns into once Gemma 4's chat template ([source](https://huggingface.co/google/gemma-4-26B-A4B-it?chat_template=default)) has rendered it:

```md
<bos><|turn>user
What is the capital of France?<turn|>
<|turn>model
The capital of France is **Paris**.<turn|>
```

Notice the special tokens like `<|turn>user`, `<turn|>`, and `<|turn>model` marking where each turn starts and ends — that's how the model tells your instructions apart from its own previous replies. You'll never write these by hand; the Chat Completions API builds this for you from your `messages` list. But it explains *why* the raw Completions API from Section 2 isn't simply interchangeable with the Chat API for a chat-tuned model: without this exact turn structure, the model doesn't reliably know who's "speaking".

You can also add a `"system"` message to set the model's overall behavior *before* the conversation starts, and continue a conversation by simply appending more `"user"`/`"assistant"` messages to the list — this is how multi-turn chat works under the hood.

In [40]:
response = client.chat.completions.create(
    model=LLM_REASONING,
    messages=[
        {"role": "system", "content": "You are a very concise assistant. Always answer in a single short sentence."},
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.0,
)

print(response.choices[0].message.content)

The capital of France is Paris.


## 2. Using the (Legacy) Completions API

Before chat-style models became standard, LLMs were used through a simpler **Completions API**: you give the model a single raw text `prompt`, and it predicts how that text continues — literally next-token prediction, with no notion of "roles" or a conversation.

This style is unusual for everyday chat use, but it's still very handy for the prompting techniques below that work by having the model *continue a pattern* you started. It also gives you precise control via the `stop` parameter, which tells the model to stop generating as soon as it produces a certain string.

Since `LLM_REASONING` only understands chat-style input (see above), we switch to `LLM_MODEL` for this. The generated text is available at `response.choices[0].text`.

In [41]:
response = client.completions.create(
    model=LLM_MODEL,
    prompt="The capital of France is",
    temperature=0.0,
    max_tokens=10,
    stop="\n",
)

print(response.choices[0].text)

Paris! 🇫🇷 


Notice the difference to Section 1: instead of *answering* the question like a chat assistant, the model simply *continues the sentence* we started — because that's literally all this API does. `stop` accepts one string or a list of strings; as soon as the model generates one of them, generation halts immediately, which helps prevent it from rambling on past the answer you actually wanted.

## 3. Prompting Strategies

Now that we can call the LLM in both styles, let's look at concrete **prompting strategies**: techniques for phrasing your prompt to get better results, without changing the model itself. Each example below calls the API directly (no wrapper functions), so you can see exactly which endpoint, model, and parameters are used for that particular technique. We'll mostly use `LLM_MODEL` here, since it works with both APIs — feel free to swap in `LLM_REASONING` (via the Chat Completions API) in any of these cells to see how a larger model performs on the same prompt.

### 3.1 Zero-shot Prompting

**Zero-shot prompting** means you directly ask the LLM to perform a task — without giving it any examples of how to do it. The model relies purely on the instructions in your prompt and the knowledge from its training. This often works remarkably well for clearly formulated tasks, especially with instruction-tuned models.

In [42]:
prompt = """Classify the sentiment of the following product review as exactly one word: positive, negative, or neutral.

Review: "The delivery was super fast and the packaging was excellent!"
Sentiment:"""

response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,
)

print(response.choices[0].message.content)

Positive



### 3.2 Few-shot Prompting

**Few-shot prompting** means you provide the LLM with a few examples directly inside the prompt to give it more context about the exact task and output format you want. This helps the model understand your query and generate more relevant, consistently-formatted answers than zero-shot alone.

In this context, the paper **"Language Models are Few-Shot Learners"** by Brown et al. (2020), which introduced GPT-3, is of particular significance. Instead of retraining the model, it is sufficient to provide a few examples directly in the prompt (few-shot) to solve new tasks. This can be described as a paradigm shift: instead of fine-tuning a model for each task, large models can generalize through prompt engineering.

[Link to the paper](https://proceedings.neurips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)

In [43]:
# In the following example, a prompt is used to extract sentiment elements from sentences.
# The model should identify the relevant elements and classify their sentiment.
prompt = '''You are a sentiment analysis model.
You will be given sentences and your task is to extract the sentiment elements from each sentence.
A sentiment element consists of a word or phrase that expresses a positive or negative sentiment.
For each sentence, you should output the sentence followed by a list of sentiment elements in the format: [("element", "sentiment")], where "element" is the word or phrase and "sentiment" is either "positive" or "negative".
If there are no sentiment elements in the sentence, output an empty list.

Sentence: It was really great in Berlin.
Sentiment Elements: [("Berlin", "positive")]
Sentence: The food wasn't very tasty.
Sentiment Elements: [("food", "negative")]
Sentence: Here in Portugal there are fantastic beaches but unfortunately the weather was bad.
Sentiment Elements: [("beaches", "positive"), ("weather", "negative")]
Sentence: The city is very beautiful.
Sentiment Elements: '''

response = client.completions.create(
    model=LLM_MODEL,
    prompt=prompt,
    temperature=0.0,
    max_tokens=50,
    stop="\n",  # stop right after the model completes this one line
)

print(response.choices[0].text)

[("city", "positive"), ("beautiful", "positive")]


### 3.3 Chain of Thought (CoT)

Introduced by [Wei et al. (2022)](https://arxiv.org/abs/2201.11903), **Chain-of-Thought (CoT) prompting** encourages the model to produce intermediate reasoning steps before giving its final answer, instead of jumping straight to a conclusion. Historically, this had to be triggered explicitly — famously, just appending the phrase *"Let's think step by step"* to a prompt could noticeably improve accuracy on tasks that require multiple reasoning steps, like arithmetic or logic puzzles.

Let's see this classic trick in action with `LLM_MODEL`, a regular (non-reasoning) model: we compare asking for the answer directly versus asking it to reason step by step first, using the exact same question both times.

In [44]:
prompt_direct = "A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there? Answer with just a number."

response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": prompt_direct}],
    temperature=0.0,
)

print(response.choices[0].message.content)

8



The correct answer is **4** (half of 16 balls = 8 golf balls, half of those = 4 blue golf balls). Without any reasoning, smaller models can easily get this wrong — for example by dividing 16 by 2 twice in the wrong way. Now let's add the classic CoT trigger phrase *"Let's think step by step"* to the exact same question:

In [45]:
prompt_cot = "A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there?\nLet's think step by step."

response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": prompt_cot}],
    temperature=0.0,
)

print(response.choices[0].message.content)

Here's how to solve this problem:

*   **Golf balls:** The juggler has 16 balls, and half are golf balls, so there are 16 / 2 = 8 golf balls.
*   **Blue golf balls:** Half of the golf balls are blue, so there are 8 / 2 = 4 blue golf balls.

**Answer:** There are 4 blue golf balls.


#### Reasoning models: thinking is no longer just a prompting trick

More recently, many LLMs are trained from the ground up to reason step by step before answering — these are called **reasoning models** (or "thinking models"), and `LLM_REASONING` (`gemma4:26b`) is one of them. For these models, you don't need to explicitly ask for reasoning with a phrase like "Let's think step by step" — **thinking is turned on by default**. The model automatically produces an internal reasoning trace before its final answer, and this trace is returned *separately* from the final answer, so you can inspect it (or hide it from an end user).

With Ollama, this reasoning trace shows up as an extra `reasoning` field on the response message (see the [Ollama blog post on thinking](https://ollama.com/blog/thinking) for more details). Let's ask `LLM_REASONING` the exact same question as above — with no CoT trick in the prompt at all:

In [46]:
response = client.chat.completions.create(
    model=LLM_REASONING,
    messages=[{"role": "user", "content": prompt_direct}],  # the plain question, no "Let's think step by step"
    temperature=0.0,
)

print("Reasoning:", response.choices[0].message.reasoning)
print("\nFinal answer:", response.choices[0].message.content)

Reasoning: *   Total balls: 16
    *   Half of the balls are golf balls: $16 / 2 = 8$ golf balls.
    *   Half of the golf balls are blue: $8 / 2 = 4$ blue golf balls.

    *   Number of blue golf balls = 4.

Final answer: 4


#### Revisiting the raw prompt: where does `reasoning` actually come from?

Remember the raw chat template from Section 1? Reasoning models extend it with a dedicated **thinking channel**, marked by `<|channel>thought ... <channel|>`, placed right before the model's actual reply — plus a `<|think|>` marker added to a hidden system turn at the very start of the prompt, telling the model that thinking is expected. Rendering our exact exchange above (question, reasoning, final answer) through Gemma 4's chat template looks like this:

```md
<bos><|turn>system
<|think|>
<turn|>
<|turn>user
A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there?<turn|>
<|turn>model
<|channel>thought
Half of 16 is 8 golf balls. Half of 8 is 4 blue golf balls.
<channel|>There are 4 blue golf balls.<turn|>
```

This is exactly why `reasoning` comes back as a separate field from `content`: the server extracts everything between `<|channel>thought` and `<channel|>` and hands it to you as `message.reasoning`, and only the text after `<channel|>` becomes `message.content`.

Even though we never asked for it, the model reasoned through the problem on its own and reached the correct answer. This "thinking" step generates extra tokens (and therefore costs extra time), so it can also be turned off — for example via the `reasoning_effort` parameter:

In [47]:
response = client.chat.completions.create(
    model=LLM_REASONING,
    messages=[{"role": "user", "content": prompt_direct}],
    temperature=0.0,
    reasoning_effort="none",  # disable thinking
)

# getattr() with a default, since the "reasoning" field is simply absent from the response now (not just empty)
print("Reasoning:", getattr(response.choices[0].message, "reasoning", None))
print("Final answer:", response.choices[0].message.content)

Reasoning: None
Final answer: 4


Under the hood, `reasoning_effort="none"` simply makes the server insert an *empty* thought channel (`<|channel>thought\n<channel|>`) right after `<|turn>model` in the raw prompt, telling the model to skip straight to the final answer without ever using the channel for content — that's why `message.reasoning` is empty this time.

So, do we still need to write "Let's think step by step" for `LLM_REASONING`? No — and adding it typically won't help any further, since the model already reasons internally by default regardless of how you phrase the prompt. The classic CoT trick above remains useful for models that don't reason by default, like `LLM_MODEL`.

### 3.4 Self-Consistency / Majority Vote

Self-Consistency is a technique introduced in [Wang et al. (2022)](https://arxiv.org/abs/2203.11171). It leverages the fact that LLMs can generate multiple different, plausible answers to the same question (e.g. by sampling with a higher `temperature`). Instead of relying on a single answer, Self-Consistency generates several independent answers and picks the most frequent one — which tends to be more reliable than trusting any single sample.

In [48]:
import re

prompt = '''Q: Anna has 2 apples. She gets 3 more apples from her friend. How many apples does she have now?
A: Anna has 2 apples. She gets 3 more. 2 + 3 = 5. The answer is 5.

Q: Tom has 7 colored pencils. He buys 5 more. How many colored pencils does he have in total?
A: Tom has 7 colored pencils. He buys 5 more. 7 + 5 = 12. The answer is 12.

Q: Lisa has 10 euros. She gets 4 euros pocket money. How much money does she have afterwards?
A: Lisa has 10 euros. She gets 4 euros more. 10 + 4 = 14. The answer is 14.

Q: Paul has 3 chocolate bars. He gets 6 chocolate bars from his brother. How much chocolate does Paul have?
A: '''

# Generate 5 independent samples with a higher temperature, each with a different seed for variety.
# No "stop" needed here - unlike the raw Completions API, a chat model already knows when its
# answer is complete and stops on its own.
predictions = []
for i in range(5):
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
        seed=i,
    )
    predictions.append(response.choices[0].message.content)

# Extract the predicted number from each sample (we take the last number mentioned as the final answer)
prediction_ints = []
for p in predictions:
    numbers = re.findall(r'\d+', p)
    prediction_ints.append(int(numbers[-1]) if numbers else None)

print("Individual predictions:", prediction_ints)
print("Majority vote:", max(set(prediction_ints), key=prediction_ints.count))

Individual predictions: [9, 9, 9, 9, 9]
Majority vote: 9


## Exercises

### Exercise 1: Few-Shot Prompting for Text Classification

Write a few-shot prompt to ask the LLM to classify the sentiment (positive, negative, neutral) of a given text. Use at least three examples in your prompt.

In [49]:
# Your code here...

<details>
<summary><b>Show Solution</b></summary>

```python
prompt = '''Sentence: It was really great in Berlin.
Label: positive
Sentence: The food wasn't very tasty.
Label: negative
Sentence: Here in Portugal there are fantastic beaches.
Label: positive
Sentence: Unfortunately the weather was bad.
Label: '''

response = client.completions.create(
    model=LLM_MODEL,
    prompt=prompt,
    temperature=0.0,
    max_tokens=10,
    stop="\n",
)

print("Label:", response.choices[0].text)
```

</details>

### Exercise 2: Named Entity Recognition with Few-Shot Learning

#### Objective
Develop a few-shot prompt strategy for the automatic recognition of **location names** (LOC - Location Entities) in English texts.  
The prepared file **`LOC_sentences.txt`** serves as the data basis, which contains sentences in the format  

```
Sentence####["Entity1", "Entity2", ...]
```

Each entry consists of an English sentence and the associated list of all location names (LOC entities) in that sentence.  

#### Requirements
1. **Few-Shot Learning:** Create a prompt with **five random few-shot examples** from `LOC_sentences.txt` that demonstrate the desired input-output behavior.  
2. **Output Format:** The model should return recognized location names as a string list (e.g., `["Berlin", "Munich"]`).  
3. **Evaluation:** Test your few-shot prompting on 64 random examples from `LOC_sentences.txt`. 

#### Evaluation with Accuracy
To evaluate the results, **accuracy** should be calculated.  

A sentence is considered **correctly labeled** if the predicted list of location names exactly matches the gold standard list from `LOC_sentences.txt`.

#### Example of Expected Behavior

```md
Input: "In Essen the two brothers met."
Output: ["Essen"]
```

Tip: You can use `eval()` to convert the model's string output into a Python list.

#### Example Prompt

```text
Recognize all location names in English sentences and return them as a list.

Sentence: The renewed administrative reform of 1815 once again brought a new district assignment for Passenheim.
Location names: ['Passenheim']

Sentence: The main base was relocated from Alderney to Jersey in 2006.
Location names: ['Alderney', 'Jersey']

Sentence: It initially runs along Bayernstraße, crosses it, and reaches the Dutzendteich stop, where there is an interchange to the S-Bahn to Altdorf.
Location names: ['Bayernstraße', 'Dutzendteich', 'Altdorf']

Sentence: Rhein-Kreis Neuss Actually, the topic always comes up at the beginning of the year when the parliamentary groups discuss the budget: Should the Rhein-Kreis Neuss sell its RWE shares?
Location names: ['Rhein-Kreis Neuss', 'Rhein-Kreis Neuss']

Sentence: A Goevier is on display at the Flugwerft Schleißheim.
Location names: ['Schleißheim']

Sentence: Today, the course of the northwestern wall ring adjoining the tower is marked by a red brick stripe, while the southwestern wall ring is still intact up to the grounds of the former Franciscan monastery St. Johannis.
Location names:
```

In [50]:
# Load the dataset
dataset = []

with open("content/LOC_sentences.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        sentence, entities_str = line.split("####")
        entities = eval(entities_str)
        dataset += [[sentence, entities]]

print("Sentence:", dataset[0][0])
print("Entities:", dataset[0][1])

# 5 random few-shot examples
import random
random.seed(42)
few_shot_examples = random.sample(dataset, 5)
test_data = [item for item in dataset if item not in few_shot_examples][:64]

Sentence: Today, the course of the northwestern wall ring adjoining the tower is marked by a red brick stripe, while the southwestern wall ring is still intact up to the grounds of the former Franciscan monastery St. Johannis.
Entities: ['St . Johannis']


In [51]:
# Your code here...

<details>
<summary><b>Show Solution</b></summary>

```python
# Create prompt
def create_prompt(examples, test_sentence):
    prompt = "Recognize all location names in English sentences and return them as a list.\n\n"
    
    for sentence, entities in examples:
        prompt += f"Sentence: {sentence}\nLocation names: {entities}\n\n"
    
    prompt += f"Sentence: {test_sentence}\nLocation names:"
    return prompt

# Evaluation
def evaluate(test_data):
    correct = 0
    total = len(test_data)
    
    for sentence, true_entities in test_data:
        prompt = create_prompt(few_shot_examples, sentence)
        response = client.completions.create(
            model=LLM_MODEL,
            prompt=prompt,
            temperature=0.0,
            max_tokens=256,
            stop="\n",
        )
        
        # Since the LLM sometimes returns invalid Python expressions, we use try-except
        try:
            predicted_entities = eval(response.choices[0].text.strip())
            if set(predicted_entities) == set(true_entities): # Use set to ignore order
                correct += 1
        except:
            pass
    
    accuracy = correct / total
    return accuracy

# Execution
accuracy = evaluate(test_data) * 100
print(f"Accuracy: {accuracy:.2f}")
```

</details>